In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, 
from sklearn.metrics import classification_report
import optuna
import ast

filePath = '../data/processed/dataset_80_20_normalized_tcr_with_cohort.csv'
df = pd.read_csv(filePath)

In [28]:
X = df.select_dtypes(include=['float64', 'int64']).drop(columns=['immunogenicity'])
y = df['immunogenicity']

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.ensemble import RUSBoostClassifier
from sklearn.model_selection import cross_val_score
from sklearn.exceptions import ConvergenceWarning
import numpy as np
import warnings

# Suprimir warnings específicos
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

def objective(trial):
    # Parámetros principales de RUSBoost
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.1, 1.0)
    sampling_strategy = trial.suggest_categorical('sampling_strategy', ['not minority', 'all'])
    replacement = trial.suggest_categorical('replacement', [True, False])

    # Elegir tipo de estimador base
    estimator_type = trial.suggest_categorical('estimator_type', ['decision_tree', 'logistic_regression'])

    if estimator_type == 'decision_tree':
        # Parámetros más robustos para Decision Tree
        max_depth = trial.suggest_int('max_depth', 4, 10)
        min_samples_split = trial.suggest_int('min_samples_split', 10, 50)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 20)

        base_estimator = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
    else:
        # Parámetros para Logistic Regression
        C = trial.suggest_float('C', 0.1, 10.0)
        solver = 'liblinear'  # Más estable para datasets pequeños/medianos

        base_estimator = LogisticRegression(
            C=C,
            solver=solver,
            max_iter=2000,  # Aumentar iteraciones
            random_state=42
        )

    clf = RUSBoostClassifier(
        estimator=base_estimator,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        sampling_strategy=sampling_strategy,
        replacement=replacement,
        random_state=42,
    )

    try:
        # Usar error_score='raise' para debuggear mejor
        scores = cross_val_score(
            clf,
            X_train,
            y_train,
            cv=5,
            scoring='roc_auc',
            n_jobs=1,
            error_score='raise'
        )

        # Verificar que los scores sean válidos
        if np.isnan(scores).any() or np.mean(scores) <= 0:
            return 0.5  # Valor por defecto en lugar de 0.0

        return np.mean(scores)

    except ValueError as e:
        if "worse than random" in str(e):
            return 0.5  # Si es peor que aleatorio, retornar 0.5 (random)
        else:
            print(f"ValueError en trial {trial.number}: {e}")
            return 0.5

    except Exception as e:
        print(f"Error en trial {trial.number}: {e}")
        return 0.5  # Retornar 0.5 en caso de cualquier otro error

In [31]:
import os


def load_best_trial_from_csv(filename='../results/reports/Optuna_Results/optuna_results_general.csv', metric='roc_auc', model_name=None):
    if not os.path.exists(filename):
        return None

    try:
        df_results = pd.read_csv(filename)

        # Aplicar filtros
        if model_name is not None:
            df_results = df_results[df_results['model'] == model_name]

        filtered = df_results[df_results['metric'] == metric]

        if filtered.empty:
            return None

        # Obtener mejor resultado
        best_entry = filtered.sort_values(by='result', ascending=False).iloc[0]
        best_params = ast.literal_eval(best_entry['best_params'])

        return optuna.trial.FixedTrial(best_params)

    except Exception:
        # Cualquier error (archivo corrupto, formato incorrecto, etc.)
        return None

In [32]:
import src.utils.Optuna_Utils as ou

best_trial = load_best_trial_from_csv(model_name='RUSBoostClassifier')

study = optuna.create_study(direction='maximize')

if best_trial is not None:
    study.enqueue_trial(best_trial.params)

study.optimize(objective, n_trials=100)  # ejecuta 200 pruebas o más

print('Mejores parámetros:', study.best_params)
print('Mejor AUC:', study.best_value)

ou.save_optuna_result(
    model_name='RUSBoostClassifier',
    metric='roc_auc',
    value=study.best_value,
    best_params=study.best_params,
    filepath=filePath
)

[I 2025-10-29 10:54:26,961] A new study created in memory with name: no-name-90e4e39c-aedc-48ea-9e02-2887f2895d1a
[I 2025-10-29 10:54:59,742] Trial 0 finished with value: 0.726508969487701 and parameters: {'n_estimators': 197, 'learning_rate': 0.872642045441107, 'sampling_strategy': 'all', 'replacement': False, 'estimator_type': 'decision_tree', 'max_depth': 10, 'min_samples_split': 36, 'min_samples_leaf': 11}. Best is trial 0 with value: 0.726508969487701.
[I 2025-10-29 10:55:02,111] Trial 1 finished with value: 0.5983076922041033 and parameters: {'n_estimators': 267, 'learning_rate': 0.2689840100242097, 'sampling_strategy': 'not minority', 'replacement': True, 'estimator_type': 'logistic_regression', 'C': 8.968771862108198}. Best is trial 0 with value: 0.726508969487701.
[I 2025-10-29 10:55:03,389] Trial 2 finished with value: 0.578778247364035 and parameters: {'n_estimators': 207, 'learning_rate': 0.3442124599034335, 'sampling_strategy': 'not minority', 'replacement': False, 'estima

Mejores parámetros: {'n_estimators': 287, 'learning_rate': 0.12259427084980129, 'sampling_strategy': 'not minority', 'replacement': True, 'estimator_type': 'decision_tree', 'max_depth': 8, 'min_samples_split': 37, 'min_samples_leaf': 7}
Mejor AUC: 0.7642576782895092


In [33]:
# Entrenar el modelo con los mejores parámetros encontrados optuna
best_params = study.best_params
estimator_type = best_params.pop('estimator_type')
if estimator_type == 'decision_tree':
    base_estimator = DecisionTreeClassifier(
        max_depth=best_params.pop('max_depth'),
        min_samples_split=best_params.pop('min_samples_split'),
        min_samples_leaf=best_params.pop('min_samples_leaf'),
        random_state=42
    )
else:
    base_estimator = LogisticRegression(
        C=best_params.pop('C'),
        solver='liblinear',
        max_iter=2000,
        random_state=42
    )
rusboost = RUSBoostClassifier(
    estimator=base_estimator,
    **best_params,
    random_state=42,
)
rusboost.fit(X_train, y_train)

,estimator,DecisionTreeC...ndom_state=42)
,n_estimators,287
,learning_rate,0.12259427084980129
,algorithm,'deprecated'
,sampling_strategy,'not minority'
,replacement,True
,random_state,42
,criterion,'gini'
,splitter,'best'
,max_depth,8
,min_samples_split,37


In [34]:
y_pred = rusboost.predict(X_test)

In [35]:
cm = confusion_matrix(y_test, y_pred, labels=rusboost.classes_)
print(cm)

[[290 137]
 [103 316]]


In [36]:
#Reporte de clasificación
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.74      0.68      0.71       427
           1       0.70      0.75      0.72       419

    accuracy                           0.72       846
   macro avg       0.72      0.72      0.72       846
weighted avg       0.72      0.72      0.72       846

